# Notebook 6: Inferens via joint sandsynligheder og diskrete SFS-observationer

I denne notebook vil jeg undersøge Bayesiansk inferens baseret på **joint sandsynligheder** den eksakte sandsynlighedstabel over kombinationer af diskrete mutationshændelser (singletons, doubletons osv.). Dette er fundamentalt anderledes fra TMRCA-inferens: i stedet for kontinuerte absorptionstider bruger jeg tælle-data om, hvilken kategori hver mutation falder i.

Via *joint_prob_graph* konstrueres en udvidet graf, der tracker alle kombinationer af diskrete hændelser (mutationer) op til en øvre grænse. Absorptionstilstandene i denne graf svarer til specifikke observationstyper, og SVGD kan da optimere mod den observerede fordelingen.

Hvad er informationsindholdet i joint SFS frem for marginal SFS? Hvornår er modellens joint-sandsynlighedstabel en god nok approksimation, og hvad er effekten af *tot_reward_limit*?

In [ ]:
from phasic import (
    Graph, with_ipv,
    GaussPrior, HalfCauchyPrior, DataPrior,
    Adam, ExpStepSize, ExpRegularization, clear_caches,
    StateIndexer, Property,
)  # ALWAYS import phasic first
import numpy as np
import jax.numpy as jnp
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
from functools import partial
from itertools import combinations_with_replacement
all_pairs = partial(combinations_with_replacement, r=2)
%config InlineBackend.figure_format = 'svg'

np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (10, 4)

clear_caches()

# Hjælpefunktion til sampling af joint-observationer
def sample_joint_observations(joint_prob_graph, theta, nr_observations=1000):
    joint_prob_graph.update_weights(theta)
    table = joint_prob_graph.joint_prob_table()
    p = table['prob'] / table['prob'].sum()
    idx = np.random.choice(table.index.values, nr_observations, p=p.to_numpy())
    return table.loc[idx, table.columns[:-1]].to_numpy().tolist()

print("Setup færdig.")

In [ ]:
# Standard koalescent med StateIndexer 
nr_samples = 4
indexer = StateIndexer(
    lineage=[Property('descendants', min_value=1, max_value=nr_samples)]
)

@with_ipv([nr_samples]+[0]*(nr_samples-1))
def coalescent_1param(state):
    transitions = []
    for i, j in all_pairs(indexer.lineage):
        p1 = indexer.lineage.index_to_props(i)
        p2 = indexer.lineage.index_to_props(j)
        same = int(i == j)
        if same and state[i] < 2: continue
        if not same and (state[i] < 1 or state[j] < 1): continue
        new = state.copy()
        new[i] -= 1; new[j] -= 1
        k = indexer.lineage.props_to_index(descendants=p1.descendants+p2.descendants)
        new[k] += 1
        transitions.append([new, [state[i]*(state[j]-same)/(1+same)]])
    return transitions

graph = Graph(coalescent_1param)
graph.plot()

## Del 1 – Effekten af tot_reward_limi` på sandsynlighedsdækning

*tot_reward_limit* sætter en øvre grænse for det samlede antal mutationer, der spores i joint-grafen. Observationer der overskrider grænsen sendes til en 'trash'-tilstand og bidrager ikke til sandsynlighedstabellen. Dette skaber et *deficit* sandsynligheder der mangler i tabellen.

### Hypotese 1

- Deficitet falder hurtigt med *tot_reward_limit* for lave mutationsrater og store θ, men langsomt for høje mutationsrater. For *tot_reward_limit=3* og mutationsrate=1 vil deficitet være under 5%, mens det ved mutationsrate=5 vil overstige 30%. Inferencekvaliteten (SVGD-estimat bias) er direkte korreleret med deficitet.

In [ ]:
# Deficit som funktion af tot_reward_limit og mutationsrate
mutation_rates = [0.5, 1.0, 2.0, 5.0]
reward_limits  = [1, 2, 3, 4, 5, 7]
true_theta_rate = [7.0]  # koalescensrate

deficit_rows = []
for mu in mutation_rates:
    for lim in reward_limits:
        jp_graph = graph.joint_prob_graph(indexer, tot_reward_limit=lim, mutation_rate=mu)
        jp_graph.update_weights(true_theta_rate + [mu])
        table = jp_graph.joint_prob_table()
        deficit = float(1 - table['prob'].sum())
        deficit_rows.append({'mu': mu, 'tot_reward_limit': lim, 'Deficit': round(deficit, 5),
                             'n_observationer i tabel': len(table)})

df_deficit = pd.DataFrame(deficit_rows)
print(df_deficit.to_string(index=False))

In [ ]:
# Visualiser deficit
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for mu, grp in df_deficit.groupby('mu'):
    axes[0].semilogy(grp['tot_reward_limit'], grp['Deficit'], 'o-', label=f'μ={mu}')
    axes[1].plot(grp['tot_reward_limit'], grp['n_observationer i tabel'], 's-', label=f'μ={mu}')

axes[0].axhline(0.05, color='red', linestyle='--', label='5% grænse')
axes[0].set_xlabel('tot_reward_limit'); axes[0].set_ylabel('Deficit (log)')
axes[0].set_title('Deficit som funktion af reward limit og μ')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('tot_reward_limit'); axes[1].set_ylabel('Antal tabelposter')
axes[1].set_title('Sandsynlighedstabelstørrelse')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Del 2 – Joint SFS vs. marginal SFS: Informationsindhold

Marginal SFS giver os det forventede antal singletons, doubletons osv. separat. Joint SFS giver mig den samlede sandsynlighed for specifikke kombinationer (fx '1 singleton og 2 doubletons'). Jeg undersøger, om joint-inferens giver smallere posterior end marginal-inferens.

### Hypotese 2

- Joint SFS-inferens (via *joint_prob_graph*) giver en smallere og bedre kalibreret posterior for θ end marginal inferens fra TMRCA-observationer, fordi joint-observationer indeholder information om kovariansen mellem SFS-komponenter. Gevinsten er størst for lav θ (færre mutationer → joint-information er mere diskriminerende).

In [ ]:
mu = 1.0
N_OBS = 1000
step = ExpStepSize(first_step=0.05, last_step=0.005, tau=40.0)

jp_graph = graph.joint_prob_graph(indexer, tot_reward_limit=3, mutation_rate=mu)

results_compare = []
theta_values = [2.0, 5.0, 10.0, 20.0]  # Varierende sand θ

for true_theta in theta_values:
    # 1. Joint-observationer
    obs_joint = sample_joint_observations(jp_graph, [true_theta, mu], N_OBS)
    
    # 2. TMRCA-observationer
    graph.update_weights([true_theta])
    obs_tmrca = graph.sample(N_OBS)
    
    # Prior via DataPrior
    prior_jp   = DataPrior(jp_graph, obs_joint)
    prior_tmrca = GaussPrior(ci=[true_theta*0.3, true_theta*3])
    
    # SVGD: joint
    sv_joint = jp_graph.svgd(
        obs_joint,
        fixed=[(1, mu)],
        prior=prior_jp,
        learning_rate=step,
        n_iterations=300
    )
    res_j = sv_joint.get_results()
    
    # SVGD: TMRCA
    sv_tmrca = graph.svgd(
        obs_tmrca,
        prior=prior_tmrca,
        learning_rate=step,
        n_iterations=300
    )
    res_t = sv_tmrca.get_results()
    
    results_compare.append({
        'Sand θ': true_theta,
        'Joint mean': round(res_j['theta_mean'][0].item(), 3),
        'Joint std':  round(res_j['theta_std'][0].item(), 4),
        'TMRCA mean': round(res_t['theta_mean'].item(), 3),
        'TMRCA std':  round(res_t['theta_std'].item(), 4),
    })
    print(f"θ={true_theta}: Joint std={res_j['theta_std'][0].item():.4f}  TMRCA std={res_t['theta_std'].item():.4f}")

df_compare = pd.DataFrame(results_compare)
print("\n", df_compare.to_string(index=False))

In [ ]:
# Visualiser sammenligning
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(df_compare['Sand θ'], df_compare['Joint mean'], 'o-', label='Joint SFS')
axes[0].plot(df_compare['Sand θ'], df_compare['TMRCA mean'], 's-', label='TMRCA')
axes[0].plot(df_compare['Sand θ'], df_compare['Sand θ'], 'k--', label='Sand θ')
axes[0].set_xlabel('Sand θ'); axes[0].set_ylabel('Posterior mean'); axes[0].legend()
axes[0].set_title('Posterior mean: Joint vs. TMRCA')

axes[1].plot(df_compare['Sand θ'], df_compare['Joint std'], 'o-', label='Joint SFS')
axes[1].plot(df_compare['Sand θ'], df_compare['TMRCA std'], 's-', label='TMRCA')
axes[1].set_xlabel('Sand θ'); axes[1].set_ylabel('Posterior std')
axes[1].set_title('Posterior usikkerhed: Joint vs. TMRCA')
axes[1].legend()
plt.suptitle('Informationsindhold: Joint SFS vs. TMRCA-observationer', fontsize=12)
plt.tight_layout(); plt.show()

## Del 3 – Joint sandsynlighedstabellens struktur og det 2D-mønster

Jeg undersøger, hvordan joint-sandsynlighedstabellen afhænger af θ. Specielt: Ændrer den relative sandsynlighed for (singleton, doubleton)-kombinationer sig med θ på en måde, der giver stærkt diskriminerende signal?

### Hypotese 3

- Heatmapmet over joint-(1-ton, 2-ton)-sandsynligheder ændrer formen systematisk med θ: lav θ giver koncentration langs diagonalen (få mutationer, men de er korrelerede), mens høj θ giver en mere symmetrisk, rektangulær fordeling. Denne form-ændring indeholder mere information om θ end summen af marginalerne.

In [ ]:
mu_heatmap = 1.0
jp_graph_hm = graph.joint_prob_graph(indexer, tot_reward_limit=4, mutation_rate=mu_heatmap)

theta_heatmap = [2.0, 7.0, 20.0]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, theta in zip(axes, theta_heatmap):
    jp_graph_hm.update_weights([theta, mu_heatmap])
    table = jp_graph_hm.joint_prob_table()
    
    # 2D pivot: 1-ton vs 2-ton
    ton_cols = ['descendants_1', 'descendants_2']
    if all(c in table.columns for c in ton_cols):
        plot_df = table[ton_cols + ['prob']].groupby(ton_cols).sum().reset_index()
        pivot = plot_df.pivot(index='descendants_2', columns='descendants_1', values='prob').fillna(0)
        sns.heatmap(pivot, ax=ax, cmap='viridis', norm=LogNorm(),
                    cbar_kws={'label': 'P(obs) (log)'})
        ax.set_title(f'θ={theta}')
        ax.set_xlabel('Antal 1-ton mutationer')
        ax.set_ylabel('Antal 2-ton mutationer')
        ax.invert_yaxis()

plt.suptitle('Joint sandsynlighed (1-ton, 2-ton) for varierende θ', fontsize=12)
plt.tight_layout(); plt.show()

## Del 4 – Sample size-effekt på joint-inferens

### Hypotese 4

- Joint SFS-inferens konvergerer til sand θ med $O(1/\sqrt{n})$ præcision ligesom TMRCA-inferens, men posterior-bredden er smallere for joint-inferens allerede ved $n=100$ observationer. Gevinsten er proportional med joint-tabelstørrelsen — store tabeller (høj *tot_reward_limit*) giver mere information per observation.**

In [ ]:
true_theta_ss = 7.0
mu_ss = 1.0
jp_ss = graph.joint_prob_graph(indexer, tot_reward_limit=3, mutation_rate=mu_ss)
graph.update_weights([true_theta_ss])

sample_sizes_joint = [50, 100, 250, 500, 1000, 3000]
step_ss = ExpStepSize(first_step=0.05, last_step=0.005, tau=30.0)
rows_ss = []

for n in sample_sizes_joint:
    # Joint-observationer
    obs_j = sample_joint_observations(jp_ss, [true_theta_ss, mu_ss], n)
    obs_t = graph.sample(n)
    
    prior_j = DataPrior(jp_ss, obs_j)
    prior_t = GaussPrior(ci=[2, 20])
    
    sv_j = jp_ss.svgd(obs_j, fixed=[(1, mu_ss)], prior=prior_j,
                       learning_rate=step_ss, n_iterations=250)
    sv_t = graph.svgd(obs_t, prior=prior_t,
                       learning_rate=step_ss, n_iterations=250)
    
    rj = sv_j.get_results(); rt = sv_t.get_results()
    rows_ss.append({
        'n': n,
        'Joint mean': round(rj['theta_mean'][0].item(), 3),
        'Joint std':  round(rj['theta_std'][0].item(), 4),
        'TMRCA mean': round(rt['theta_mean'].item(), 3),
        'TMRCA std':  round(rt['theta_std'].item(), 4),
    })
    print(f"n={n:5d}: Joint std={rj['theta_std'][0].item():.4f}  TMRCA std={rt['theta_std'].item():.4f}")

df_ss = pd.DataFrame(rows_ss)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(df_ss['n'], df_ss['Joint std'], 'o-', label='Joint SFS')
axes[0].loglog(df_ss['n'], df_ss['TMRCA std'], 's-', label='TMRCA')
# Theoretical O(1/sqrt(n)) line
n_arr = np.array(df_ss['n'], dtype=float)
axes[0].loglog(n_arr, df_ss['Joint std'].iloc[0] * np.sqrt(df_ss['n'].iloc[0]/n_arr),
               'k--', alpha=0.5, label='O(1/√n) reference')
axes[0].set_xlabel('Sample size n (log)'); axes[0].set_ylabel('Posterior std (log)')
axes[0].set_title('Konvergenshastighed: Joint vs. TMRCA')
axes[0].legend()

axes[1].semilogx(df_ss['n'], df_ss['Joint mean'], 'o-', label='Joint')
axes[1].semilogx(df_ss['n'], df_ss['TMRCA mean'], 's-', label='TMRCA')
axes[1].axhline(true_theta_ss, color='red', linestyle='--', label=f'Sand θ={true_theta_ss}')
axes[1].set_xlabel('Sample size n'); axes[1].set_ylabel('Posterior mean')
axes[1].set_title('Bias som funktion af n')
axes[1].legend()

plt.suptitle(f'Sample size-effekt (sand θ={true_theta_ss}, μ={mu_ss})', fontsize=12)
plt.tight_layout(); plt.show()

## Del 5 – To-locus ARG: Joint inferens af koalescens og rekombination

Den naturlige udvidelse er at bruge joint-sandsynligheder fra et two-locus ARG til at inferere både koalescensrate og rekombinationsrate. Vi undersøger, om joint-observationer kan adskille de to parametre.

### Hypotese 5

- To-locus joint-observationer (hvilken kombination af allel-typer ses ved de to loci) indeholder information om rekombinationsraten. Vi forventer en klart identificerbar posterior for (θ₀, R), fordi høj rekombination øger sandsynligheden for uafhængige SFS ved de to loci.

In [ ]:
# Two-locus ARG model 
nr_samples_arg = 3
indexer_arg = StateIndexer(
    descendants=[
        Property('loc1', min_value=0, max_value=nr_samples_arg),
        Property('loc2', min_value=0, max_value=nr_samples_arg)
    ]
)

initial_arg = [0] * indexer_arg.state_length
initial_arg[indexer_arg.descendants.props_to_index(loc1=1, loc2=1)] = nr_samples_arg

@with_ipv(initial_arg)
def two_locus_arg(state, indexer=None):
    transitions = []
    if state.sum() <= 1: return transitions
    for i in range(indexer.state_length):
        if state[i] == 0: continue
        pi = indexer.descendants.index_to_props(i)
        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            pj = indexer.descendants.index_to_props(j)
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            child = state.copy(); child[i] -= 1; child[j] -= 1
            loc1 = pi.loc1 + pj.loc1; loc2 = pi.loc2 + pj.loc2
            if loc1 <= nr_samples_arg and loc2 <= nr_samples_arg:
                child[indexer.props_to_index(loc1=loc1, loc2=loc2)] += 1
                transitions.append([child, [state[i]*(state[j]-same)/(1+same), 0]])
        # Rekombination
        if pi.loc1 > 0 and pi.loc2 > 0:
            child = state.copy(); child[i] -= 1
            child[indexer.props_to_index(loc1=pi.loc1, loc2=0)] += 1
            child[indexer.props_to_index(loc1=0, loc2=pi.loc2)] += 1
            transitions.append([child, [0, 1]])
    return transitions

graph_arg = Graph(two_locus_arg, indexer=indexer_arg)
print(f"ARG tilstandsrum: {graph_arg.vertices_length()} tilstande")
graph_arg.plot(max_nodes=50, nodesep=0.3)

In [ ]:
# Byg joint probability graph for ARG
mu_arg = 1.0
jp_arg = graph_arg.joint_prob_graph(
    indexer_arg,
    reward_only=['loc1', 'loc2'],
    reward_limit=1,
    tot_reward_limit=2,
    mutation_rate=mu_arg
)
print(f"Joint ARG graph: {jp_arg.vertices_length()} tilstande")

true_theta_arg = [10.0, 1.0, mu_arg]  # [koalescens, rekombination, mutation]
obs_arg = sample_joint_observations(jp_arg, true_theta_arg, nr_observations=1000)
print(f"Samplet {len(obs_arg)} observationer. Første 5: {obs_arg[:5]}")

In [ ]:
# SVGD inferens for ARG: estimér koalescens og rekombination
step_arg = ExpStepSize(first_step=0.1, last_step=0.01, tau=50.0)

sv_arg = jp_arg.svgd(
    observed_data=obs_arg,
    fixed=[(2, mu_arg)],  # Fix mutationsrate
    prior=[
        GaussPrior(ci=[5, 25]),   # θ₀ (koalescens)
        GaussPrior(ci=[0, 5]),    # θ₁ (rekombination)
        None                       # μ (fixed)
    ],
    n_particles=60,
    n_iterations=300,
    learning_rate=step_arg,
    optimizer=Adam(learning_rate=0.25),
)
sv_arg.summary()

res_arg = sv_arg.get_results()
print(f"\nSand θ: koalescens={true_theta_arg[0]}, rekombination={true_theta_arg[1]}")
print(f"Estimat: θ₀={res_arg['theta_mean'][0]:.3f}, θ₁={res_arg['theta_mean'][1]:.3f}")

In [ ]:
# Pairwise posterior for ARG
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sv_arg.plot_pairwise(true_theta=[true_theta_arg[0], true_theta_arg[1]], ax=axes[0])
axes[0].set_title('Joint posterior (θ₀=koalescens, θ₁=rekombination)')
sv_arg.plot_convergence(ax=axes[1])
axes[1].set_title('SVGD konvergens')
plt.suptitle('Two-locus ARG: joint inferens af koalescens og rekombination', fontsize=12)
plt.tight_layout(); plt.show()